# <font color=blue> 기계학습 기초: 데이터, 데이터의 시각화, 차원축소, 군집화

### 실습 주제
1. 데이터 프레임 활용
2. 데이터분포의 시각화
3. 기초 통계량과 Pearson 상관계수
4. 차원축소
5. 군집화

### 사용할 데이터:
1. Score data (100 x 4)
2. Iris (150 x 4)
1.   Gene expression 데이터 (1092 x 20531)
2.   Low quality digit images (1797 x 8 x 8)

2019.08.17 단국대학교 윤 석현

## 학습 목표
- pandas DataFrame의 기본 구조와 주요 조작 방법을 익힌다.
- 히스토그램, 산점도, pair plot으로 데이터 분포를 확인한다.
- 공분산, Pearson 상관계수, PCA, t-SNE, clustering의 기본 아이디어를 실습한다.

## 사용할 데이터
- `load_data('scores')`: 학생 점수 예제
- `seaborn` iris 데이터
- `load_data('tcga-brca')`: 유전자 발현 예제
- `sklearn` digits 데이터

## 직접 바꿔볼 것
- 시각화에 사용할 변수 이름
- PCA/t-SNE 파라미터
- clustering의 cluster 개수


### 먼저, 필요한 패키지 (추가) 설치

In [ ]:
##'''
!pip install scikit-network
!pip install statannotations
!pip install anndata
!pip install mlbi-lab --upgrade
##'''

### 필요한 패키지 로드

In [1]:
import os
import pandas as pd
import numpy as np

from sklearn.datasets import load_iris, load_digits

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from mlbi.datasets import load_data, load_anndata
from statannotations.Annotator import Annotator

## For clustering
from sklearn import cluster, mixture
from sklearn.neighbors import kneighbors_graph
from sknetwork.clustering import Louvain


/mnt/HDD2/Google_drive/PyPI_Git_Publish/pub_mlbi_lab/src/mlbi/datasets.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


### 1. pandas 데이터 프레임 활용

pandas 데이터프레임의 구조와 특징을 이해하고 이의 활용법을 익힌다.
https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html    
    
사용할 데이터: Score data (100 x 4)

In [2]:
load_data()

You can select one of:
  cancerseek
  ccle-ctrpv2
  heart_failure
  hotel_bookings
  house_price
  labor_force
  metabric
  scores
  tcga-brca
  time-series
  time-series2


#### (1) 데이터 (csv file) 불러오기 (데이터 프레임으로 저장)

In [3]:
df_score = load_data('scores')

In [4]:
df_score

,Group,English,Math,Science
0,A,77,72,75
1,A,81,67,68
2,A,74,74,72
3,A,89,64,68
4,A,78,71,74
...,...,...,...,...
95,B,75,81,77
96,B,65,84,77
97,B,75,77,73
98,B,72,82,79


In [5]:
df_score.shape

(100, 4)

In [6]:
df_score.describe()

,English,Math,Science
count,100.000000,100.000000,100.000000
mean,74.420000,75.280000,74.910000
std,5.896721,5.312611,2.667026
min,65.000000,64.000000,68.000000
25%,69.000000,71.000000,73.000000
50%,75.000000,75.500000,75.000000
75%,78.000000,80.000000,77.000000
max,89.000000,85.000000,81.000000


In [7]:
df_score.to_csv('my_score_data.csv', index = False)

In [8]:
df_score['Group'].value_counts()

Group
A    50
B    50
Name: count, dtype: int64

#### (2) DataFrame 활용 연습
1. head() method를 이용한 Data frame 확인
2. shape attribute를 이용한 size 확인
3. row name(index) 및 column name 확인
4. index/column name을 이용한 행/열 가져오기
5. pandas series
6. NA(None)이 포함된 element 확인하
7. 행/열의 합/평균/표준편차 구하기
8. 행/열의 최대값/최소값/최대값의 index 구하기
9. 범주형 데이터의 범주 카운트 구하기
10. 조건에 맞는 행들만 선택하기 - boolean indexing
11. 데이터 프레임을 csv 파일로 저장하기

In [9]:
df_score.index.values

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67,
       68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84,
       85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99])

In [10]:
df_score.columns.values

array(['Group', 'English', 'Math', 'Science'], dtype=object)

In [11]:
df_score[['Group', 'English']]

,Group,English
0,A,77
1,A,81
2,A,74
3,A,89
4,A,78
...,...,...
95,B,75
96,B,65
97,B,75
98,B,72


In [12]:
df_score.loc[5, :]

Group       A
English    77
Math       67
Science    68
Name: 5, dtype: object

In [13]:
b = df_score['Math'] > 74
df_sel = df_score.loc[b,:]
df_sel

,Group,English,Math,Science
8,A,79,75,79
23,A,78,75,77
27,A,72,76,78
47,A,76,75,79
50,B,69,83,78
51,B,76,78,75
52,B,74,76,72
53,B,67,82,74
54,B,74,80,77
55,B,71,77,70


In [14]:
df_sel.index.values

array([ 8, 23, 27, 47, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62,
       63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79,
       80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96,
       97, 98, 99])

In [15]:
df_score[['Math', 'English', 'Science']].max(axis = 0)

Math       85
English    89
Science    81
dtype: int64

In [16]:
df_score.max(axis = 0)

Group       B
English    89
Math       85
Science    81
dtype: object

In [17]:
df_score.loc[10,:]

Group       A
English    76
Math       66
Science    70
Name: 10, dtype: object

In [18]:
df_score.index.values

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67,
       68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84,
       85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99])

### 2. 데이터(분포)의 시각화

1. Histogram https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.hist.html
1. scatter plot https://matplotlib.org/3.5.1/api/_as_gen/matplotlib.pyplot.scatter.html
1. pairplot https://seaborn.pydata.org/generated/seaborn.pairplot.html

사용할 데이터: iris data

#### (1) iris 데이터 로드

In [19]:
df_iris = sns.load_dataset('iris')
# df_iris.to_csv('my_iris_data.csv')
df_iris

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica
146,6.3,2.5,5.0,1.9,virginica
147,6.5,3.0,5.2,2.0,virginica
148,6.2,3.4,5.4,2.3,virginica


In [20]:
df_iris['species'].value_counts()

species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64

In [21]:
df_iris.describe()

,sepal_length,sepal_width,petal_length,petal_width
count,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333
std,0.828066,0.435866,1.765298,0.762238
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.350000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


In [22]:
df_iris['species'].value_counts()

species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64

#### (3) 히스토그램과 산점도 그려보기
1. Histogram https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.hist.html
1. scatter plot https://matplotlib.org/3.5.1/api/_as_gen/matplotlib.pyplot.scatter.html
1. scatter plot https://seaborn.pydata.org/generated/seaborn.scatterplot.html
1. pairplot https://seaborn.pydata.org/generated/seaborn.pairplot.html

In [23]:
df = df_score

In [ ]:
df.columns.values

In [ ]:
ax = df['Science'].hist(bins = 40, alpha = 0.8, figsize = (5,4))
ax.set_xlabel('Science')
ax.set_ylabel('Frequency')
plt.show()

In [ ]:
plt.figure(figsize = (5,5))
sns.scatterplot(data = df, x = 'Math', y = 'Science', hue = 'Group')
plt.show()

In [ ]:
plt.figure(figsize=(5,5))
sns.pairplot(df, hue='Group', diag_kind  = 'kde')

In [ ]:
x_label = 'Math'
y_label = 'Science'
group = 'Group'

plt.figure(figsize = (5,5), dpi = 100)
sns.scatterplot(data = df, x = x_label, y = y_label, hue = group)

x = df[x_label]
y = df[y_label]
class_lst = list(df[group].unique())

b = df[group] == class_lst[0]
sns.kdeplot( x = x[b], y = y[b], cmap = 'Blues', bw_adjust = 2 )
sns.kdeplot( x = x[~b], y = y[~b], cmap = 'Oranges', bw_adjust = 2 )

# plt.show()

In [ ]:
def get_pairs(lst, annot_ref = None):

    lst_pair = []
    if annot_ref in lst:
        for k, l1 in enumerate(lst):
            if l1 != annot_ref:
                lst_pair.append((annot_ref, l1))
    else:
        for k, l1 in enumerate(lst):
            for j, l2 in enumerate(lst):
                if j <  k:
                    lst_pair.append((l1, l2))
    return lst_pair

group_var = 'Group'
item = 'Math'

lst_pair = get_pairs(list(df[group_var].unique()))

ax = sns.boxplot(data = df, x = group_var, y = item, hue = group_var)
sns.stripplot(data = df, x = group_var, y = item, jitter=True, color="black", ax = ax)

annotator = Annotator(ax, pairs=lst_pair, data=df, x=group_var, y=item)
annotator.configure(test='t-test_ind', text_format='simple', fontsize = 9,
                    loc='inside', verbose=False, show_test_name=False)
ax = annotator.apply_and_annotate()


### 3. 공분산 행렬과 Pearson 상관 계수 구하기

사용할 데이터: Iris (150 x 4)

In [ ]:
df_score.columns

In [ ]:
df = df_score.loc[:,df_score.columns[1:]]

In [ ]:
df.head()

In [ ]:
## 평균 (벡터) 구하기
m_x = df.mean(axis = 0)
m_x

In [ ]:
m_x.shape

In [ ]:
## 상관행렬 구하기
R_xx = df.transpose().dot(df)/df.shape[0]
R_xx

In [ ]:
## 공분산행렬 구하기
C_xx = R_xx - np.outer(m_x, m_x)

In [ ]:
C_xx

In [ ]:
df.std(axis = 0)**2

In [ ]:
## Pearson 상관계수
s1 = C_xx.loc['Math', 'Math']
s2 = C_xx.loc['English', 'English']
s12 = C_xx.loc['Math', 'English']

ro12 = s12/np.sqrt(s1*s2)
ro12

In [ ]:
from scipy.stats import pearsonr

In [ ]:
## Pearson 상관계수
pearsonr(df['Math'], df['English'])

### Homework
1. scores 데이터에 대해 pair-wise scatter plot을 그려라.
1. scores 데이터에 대해 수학-영어, 영어-과학, 수학-과학 점수간의 Pearson 상관계수를 구하라.
2. iris 데이터에 대해 4개 항목간의 Pearson 상관계수를 모두 구하라.

## 4. 차원 축소와 데이터 분포의 시각화
#####  
<div>
<img src=attachment:dim_reduction_concept.png width="500"/>   
</div>

#### Two approaches typically used
1.  PCA:  https://datascienceschool.net/view-notebook/f10aad8a34a4489697933f77c5d58e3a/ (전처리용)
2.  tSNE:  https://umap-learn.readthedocs.io/en/latest/basic_usage.html (시각화용)!

#### PCA를 적용하여 2차원 축소한 후 데이터 분포 시각화
     
<div>
<img src=attachment:image.png width="500"/>   
</div>

#### (1) 데이터 로드

In [ ]:
df_iris = sns.load_dataset('iris')
df_iris.head()

In [ ]:
cols_selected = df_iris.columns[:4].tolist()
df = df_iris.loc[:,cols_selected]
df.head()

#### (3) PCA를 통한 차원 축소: 4차원 -> 2차원으로

In [ ]:
## create PCA object
pca_2d = PCA(n_components = 2)
X_2d = pca_2d.fit_transform(df)

In [ ]:
df.shape, X_2d.shape

In [ ]:
X_2d

In [ ]:
df_2d = pd.DataFrame(X_2d, columns = ['PC1', 'PC2'])

In [ ]:
df_2d.head()

#### (4) 산점도 확인하기

In [ ]:
df_2d['species'] = df_iris['species']

In [ ]:
df_2d

In [ ]:
plt.figure(figsize = (5,5))
# sns.scatterplot(x = df_2d['PC1'], y = df_2d['PC2'], hue = df_2d['species'])
sns.scatterplot(data = df_2d, x = 'PC1', y = 'PC2', hue = 'species')
plt.show()

### t-SNE를 적용하여 2차원 축소한 후 데이터 분포 시각화 - Gene expression data

#### (1) 데이터 불러오기

In [ ]:
load_data()

In [ ]:
df_tcga = load_data('tcga-brca')

In [ ]:
df_tcga.keys()

In [ ]:
df_gene_exp = df_tcga['gene_expression']
df_clinical = df_tcga['clinical_info']

df.shape, df_clinical.shape

In [ ]:
df_clinical.columns.values

In [ ]:
df_clinical['subtype'].value_counts()

In [ ]:
## Set feature vector and target values
X = np.log10(df_gene_exp + 1)
y = list(df_clinical['subtype'])  # df.Yumor_type
target_names = list(set(y))

print('Tumor subtypes: ', target_names)

In [ ]:
X.shape

In [ ]:
len(y)

#### (2) PCA를 적용하여 2차원 축소한 후 데이터 분포 시각화

In [ ]:
X_2d_pca = pca_2d.fit_transform(X)

df_2d_pca = pd.DataFrame(X_2d_pca, columns = ['PC1', 'PC2'])
df_2d_pca['Tumor type'] = list(y)

In [ ]:
df_2d_pca

In [ ]:
plt.figure(figsize = (6,6))
sns.scatterplot(data = df_2d_pca, x = 'PC1', y = 'PC2', hue = 'Tumor type')
plt.show()

#### (3) tSNE(비선형 차원축소, 교재 12장)을 이용한 데이터 시각화

In [ ]:
X_2d_tsne = TSNE(learning_rate=3000, init='pca').fit_transform(X)
print(X.shape, X_2d_tsne.shape)

In [ ]:
df_2d_tsne = pd.DataFrame(X_2d_tsne, columns = ['D1', 'D2'])
df_2d_tsne['Tumor type'] = list(y)

In [ ]:
df_2d_tsne

In [ ]:
plt.figure(figsize = (6,6))
sns.scatterplot(data = df_2d_tsne, x = 'D1', y = 'D2', hue = 'Tumor type')
plt.show()

###  t-SNE를 적용하여 2차원 축소한 후 데이터 분포 시각화 - Digits 데이터

#### (1) 데이터 로드

In [ ]:
digits = load_digits()
print(digits.DESCR)

#### (2) 불러온 데이터 확인

In [ ]:
digits.keys()

In [ ]:
print(digits.data.shape)
print(digits.images.shape)
print(digits.target.shape)
digits.target_names

In [ ]:
## Check images
fig, ax_array = plt.subplots(4, 10)
axes = ax_array.flatten()
for i, ax in enumerate(axes):
    ax.imshow(digits.images[i], cmap='gray_r')

plt.setp(axes, xticks=[], yticks=[], frame_on=False)
plt.show()

In [ ]:
digits.target[:30]

In [ ]:
## 2-dim image data
digits.images[0]

In [ ]:
## 1-dim flattened data
digits.data[0]

In [ ]:
## Set feature vector and target values
X = digits.data
y = digits.target
target_names = list(set(y))

print('Target names: ', target_names)

In [ ]:
y.shape, X.shape

#### (3) PCA를 적용하여 2차원 축소한 후 데이터 분포 시각화

In [ ]:
## create PCA object
pca_2d = PCA(n_components=2)
X_2d_pca = pca_2d.fit_transform(X)

df_2d_pca = pd.DataFrame(X_2d_pca, columns = ['PC1', 'PC2'])
df_2d_pca['label'] = y

In [ ]:
X_2d_pca

In [ ]:
df_2d_pca

In [ ]:
df_2d_pca['label'].value_counts()

In [ ]:
X.shape, X_2d_pca.shape

In [ ]:
plt.figure(figsize = (5,5))
g = sns.scatterplot(data = df_2d_pca, x = 'PC1', y = 'PC2',
                hue = 'label', palette = 'Spectral', legend='full')
g.legend(loc='right', bbox_to_anchor=(1.2, 0.5), ncol=1)
g.set_ylabel('PC1', fontsize = 12)
g.set_ylabel('PC2', fontsize = 12)
plt.show()

#### (4) tSNE을 이용한 데이터 시각화

In [ ]:
X_2d_tsne = TSNE(learning_rate=300, init='pca').fit_transform(X)
print(X.shape, X_2d_tsne.shape)

In [ ]:
df_2d_tsne = pd.DataFrame(X_2d_tsne, columns = ['D1', 'D2'])
df_2d_tsne['label'] = y

In [ ]:
df_2d_tsne

In [ ]:
plt.figure(figsize = (5,5))
g = sns.scatterplot(data = df_2d_tsne, x = 'D1', y = 'D2',
                hue = 'label', palette = 'Spectral', legend='full')
g.legend(loc='right', bbox_to_anchor=(1.2, 0.5), ncol=1)
g.set_ylabel('D1', fontsize = 12)
g.set_ylabel('D2', fontsize = 12)
plt.show()

In [ ]:
!pip install bokeh

In [ ]:
from io import BytesIO
from PIL import Image
import base64

def embeddable_image(data):
    img_data = 255 - 15 * data.astype(np.uint8)
    image = Image.fromarray(img_data, mode='L').resize((64, 64), Image.BICUBIC)
    buffer = BytesIO()
    image.save(buffer, format='png')
    for_encoding = buffer.getvalue()
    return 'data:image/png;base64,' + base64.b64encode(for_encoding).decode()

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import HoverTool, ColumnDataSource, CategoricalColorMapper
from bokeh.palettes import Spectral10

output_notebook()
digits_df = pd.DataFrame(X_2d_tsne, columns=('x', 'y'))
digits_df['digit'] = [str(x) for x in digits.target]
digits_df['image'] = list(map(embeddable_image, digits.images))

datasource = ColumnDataSource(digits_df)
color_mapping = CategoricalColorMapper(factors=[str(9 - x) for x in digits.target_names],
                                       palette=Spectral10)

plot_figure = figure(
    title='UMAP projection of the Digits dataset',
    width=600,
    height=600,
    tools=('pan, wheel_zoom, reset')
)

plot_figure.add_tools(HoverTool(tooltips="""
<div>
    <div>
        <img src='@image' style='float: left; margin: 5px 5px 5px 5px'/>
    </div>
    <div>
        <span style='font-size: 16px; color: #224499'>Digit:</span>
        <span style='font-size: 18px'>@digit</span>
    </div>
</div>
"""))

plot_figure.circle(
    'x',
    'y',
    source=datasource,
    color=dict(field='digit', transform=color_mapping),
    line_alpha=0.6,
    fill_alpha=0.6,
    size=4
)
show(plot_figure)

## 5. 군집화 (Clustering)
#####  
<div>
<img src=attachment:clustering.png width="600"/>   
</div>

1. k-means: https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html
2. Gaussian mixture model: https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html
3. Louvain algorithm: https://scikit-network.readthedocs.io/en/latest/tutorials/clustering/louvain.html


#### (1) Load data

In [ ]:
## Iris data
iris = load_iris()
X = iris.data
y = iris.target
target_names = list(set(y))

In [ ]:
## digit image data
digits = load_digits()
X = digits.data
y = digits.target
target_names = list(set(y))

In [ ]:
X.shape, y.shape

#### (2) Dimension reduction using tSNE (for visualization)

In [ ]:
X_2d_tsne = TSNE(learning_rate=300, init='pca').fit_transform(X)
X_2d_tsne.shape, X.shape

In [ ]:
df_2d_tsne = pd.DataFrame(X_2d_tsne, columns = ['D1', 'D2'])
df_2d_tsne['target_value'] = y

In [ ]:
plt.figure(figsize = (6,6))
g = sns.scatterplot(data = df_2d_tsne, x = 'D1', y = 'D2')
plt.show()

In [ ]:
X_pca = X

In [ ]:
df_2d_tsne

#### (3a) k-means clustering

In [ ]:
N_clusters = 16
km = cluster.KMeans(n_clusters = N_clusters)
km.fit(X_pca)
df_2d_tsne['cluster_km'] = km.labels_

In [ ]:
df_2d_tsne

In [ ]:
df_2d_tsne.head()

In [ ]:
df_2d_tsne['cluster_km'].value_counts()

In [ ]:
plt.figure(figsize = (6,6))
g = sns.scatterplot(data = df_2d_tsne, x = 'D1', y = 'D2',
                hue = 'cluster_km', palette = 'Spectral', legend='full')
g.legend(loc='right', bbox_to_anchor=(1.2, 0.5), ncol=1)
plt.show()

#### (3b) GMM clustering

In [ ]:
N_clusters = 10
gmm = mixture.GaussianMixture(n_components = N_clusters, random_state = 0)
df_2d_tsne['cluster_gmm'] = gmm.fit_predict(X_pca)

In [ ]:
df_2d_tsne.head()

In [ ]:
plt.figure(figsize = (6,6))
g = sns.scatterplot(data = df_2d_tsne, x = 'D1', y = 'D2',
                hue = 'cluster_gmm', palette = 'Spectral', legend='full')
g.legend(loc='right', bbox_to_anchor=(1.25, 0.5), ncol=1)
plt.show()

#### (3c) clustering using Louvain algorithm

In [ ]:
adj = kneighbors_graph(X_pca, 15, mode='connectivity', include_self=True)
louvain = Louvain(resolution = 1)
cluster_label = louvain.fit_predict(adj)
df_2d_tsne['cluster_louvain'] = cluster_label

In [ ]:
plt.figure(figsize = (6,6))
g = sns.scatterplot(data = df_2d_tsne, x = 'D1', y = 'D2',
                hue = 'cluster_louvain', palette = 'Spectral', legend='full')
g.legend(loc='right', bbox_to_anchor=(1.2, 0.5), ncol=1)
plt.show()